In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('F1Strategy.csv')

In [ ]:
df.head()

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0,0.017241,0.117647,0.0
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0
2,ALB,3,MEDIUM,1,4.0,18,91.768,Abu Dhabi Grand Prix,2023,-1.792,-8.857,0,0,0.051724,0.235294,0.0
3,ALB,4,MEDIUM,1,5.0,18,91.591,Abu Dhabi Grand Prix,2023,-0.177,-9.034,0,0,0.068966,0.294118,0.0
4,ALB,5,MEDIUM,1,6.0,18,91.422,Abu Dhabi Grand Prix,2023,-0.169,-9.203,0,0,0.086207,0.352941,0.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101371 entries, 0 to 101370
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Driver                  101371 non-null  object 
 1   LapNumber               101371 non-null  int64  
 2   Compound                101305 non-null  object 
 3   Stint                   101371 non-null  int64  
 4   TyreLife                101371 non-null  float64
 5   Position                101371 non-null  int64  
 6   LapTime (s)             101371 non-null  float64
 7   Race                    101371 non-null  object 
 8   Year                    101371 non-null  int64  
 9   LapTime_Delta           101371 non-null  float64
 10  Cumulative_Degradation  101371 non-null  float64
 11  PitStop                 101371 non-null  int64  
 12  PitNextLap              101371 non-null  int64  
 13  RaceProgress            101371 non-null  float64
 14  Normalized_TyreLife 

In [ ]:
df["RaceId"] = df["Race"] + "_" + df["Year"].astype(str)

In [ ]:
df = df.sort_values(by=['RaceId', 'Driver', 'LapNumber'])

In [ ]:
df['prev_lap_time'] = df.groupby(['RaceId', 'Driver'])['LapTime (s)'].shift(1)

In [ ]:
df['prev_position'] = df.groupby(['RaceId', 'Driver'])['Position'].shift(1)

In [ ]:
df['avg_last_3_laps'] = (
    df.groupby(['RaceId', 'Driver'])['LapTime (s)']
    .rolling(3)
    .mean()
    .reset_index(level=[0,1], drop=True)
)

In [ ]:
df['lap_time_diff'] = df['LapTime (s)'] - df['prev_lap_time']

In [ ]:
df['pit_event'] = df['PitNextLap'].shift(1).fillna(0)
df['lap_since_last_pit'] = (
    df.groupby(['RaceId', 'Driver'])['pit_event']
    .cumsum()
)

In [ ]:
df['Total_laps'] = df.groupby(['RaceId', 'Driver'])['LapNumber'].transform('max')

In [ ]:
df['race_progress'] = df['LapNumber'] / df['Total_laps']

In [ ]:
df = df.dropna()

In [ ]:
features = [
    "LapTime (s)",
    "prev_lap_time",
    "avg_last_3_laps",
    "lap_time_diff",
    "Position",
    "prev_position",
    "lap_since_last_pit",
    "race_progress"
]

X = df[features]
y = df['PitNextLap']

In [ ]:
train = df[df['Year'] < 2024]
test = df[df['Year'] >= 2024]
X_train = train[features]
X_test = test[features]
y_train = train['PitNextLap']
y_test = test['PitNextLap']

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
print("Train size:", train.shape)
print("Test size:", test.shape)

print("Train years:", train["Year"].unique())
print("Test years:", test["Year"].unique())

Train size: (44958, 25)
Test size: (52661, 25)
Train years: [2023 2022]
Test years: [2024 2025]


In [ ]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.88      0.86     34808
           1       0.74      0.65      0.69     17853

    accuracy                           0.81     52661
   macro avg       0.79      0.77      0.78     52661
weighted avg       0.80      0.81      0.80     52661



In [ ]:
df.columns

Index(['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'Race', 'Year', 'LapTime_Delta',
       'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress',
       'Normalized_TyreLife', 'Position_Change', 'RaceId', 'prev_lap_time',
       'prev_position', 'avg_last_3_laps', 'lap_time_diff', 'pit_event',
       'lap_since_last_pit', 'Total_laps', 'race_progress'],
      dtype='object')